In [2]:
!uv pip install evaluate wandb nvidia-dali-cuda120 timm -q


In [7]:
!mkdir /kaggle/working/abc

In [6]:
import os
import math
import json
import hashlib
import random
import csv
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision.ops import sigmoid_focal_loss
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from numpy.lib.format import open_memmap

import timm
from accelerate import Accelerator
from accelerate.utils import set_seed

from nvidia.dali import fn, types, pipeline_def
from nvidia.dali.plugin.pytorch import DALIGenericIterator


# =============================================================================
# SEED / DEVICE
# =============================================================================
seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

accelerator = Accelerator()
device = accelerator.device


# =============================================================================
# CONFIG
# =============================================================================
CONFIG = {
    "batch_size": 64,
    "image_height": 120,
    "image_width": 120,
    "prefetch_queue_depth": 2,
    "time_window_in_seconds": 3,
    "frames_per_window": 16,
    "window_stride_by_seconds": 2,

    "fusion_batch_size": 256,
    "fusion_epochs": 50,
    "fusion_lr": 1e-3,
    "fusion_weight_decay": 1e-4,
    "fusion_patience": 1000,

    "use_features": True,
}

PATHS = {
    "eye_train_dirs": [
        "/kaggle/input/datasets/manith04/ddd-train1-righteye-v2",
        "/kaggle/input/datasets/manith04/ddd-train2-righteye-v2",
    ],
    "eye_val_dirs": [
        "/kaggle/input/datasets/manith04/ddd-eval-righteye-v2",
    ],
    "eye_test_dirs": [],

    "mouth_train_dirs": [
        "/kaggle/input/datasets/manith04/ddd-train1-mouth-v2",
        "/kaggle/input/datasets/manith04/ddd-train2-mouth-v2",
    ],
    "mouth_val_dirs": [
        "/kaggle/input/datasets/manith04/ddd-eval-mouth-v2",
    ],
    "mouth_test_dirs": [],

    "head_train_dirs": [
        "/kaggle/input/datasets/manith04/ddd-train1-head-v2",
        "/kaggle/input/datasets/manith04/ddd-train2-head-v2",
    ],
    "head_val_dirs": [
        "/kaggle/input/datasets/manith04/ddd-eval-head-v2",
    ],
    "head_test_dirs": [],

    "drowsiness_train_label_root_dirs": [
        "/kaggle/input/datasets/manith04/dddeeee",
        "/kaggle/input/datasets/manith04/ddewewewee",
    ],
    "drowsiness_val_label_root_dirs": [
        "/kaggle/input/datasets/manith04/gdgdfhfg",
    ],
    "drowsiness_test_label_root_dirs": [],

    "eye_ckpt": "/kaggle/input/models/manith04/ddd-manith-bbb/pytorch/default/1/best_val_accuracy_righteye_model.pth",
    "mouth_ckpt": "/kaggle/input/models/manith04/ddd-manith-bbb/pytorch/default/1/best_val_accuracy_mouth_model.pth",
    "head_ckpt": "/kaggle/input/models/manith04/ddd-manith-bbb/pytorch/default/1/best_val_accuracy_head_model.pth",

    "output_dir": "/kaggle/working/abc/fusion_outputs",
}

os.makedirs(PATHS["output_dir"], exist_ok=True)


# =============================================================================
# HELPERS
# =============================================================================
def prepare_sequence_batch(frames, labels, frames_per_window, classification_type):
    flat_batch_size = frames.shape[0]
    real_batch_size = flat_batch_size // frames_per_window

    frames = frames.view(real_batch_size, frames_per_window, *frames.shape[1:])
    labels = labels.view(real_batch_size, frames_per_window, -1)[:, -1, 0]

    if classification_type == "binary":
        labels = labels.view(-1).float()
    elif classification_type == "multiclass":
        labels = labels.view(-1).long()
    else:
        raise ValueError("classification_type must be 'binary' or 'multiclass'")

    return frames, labels


# =============================================================================
# DALI SOURCE / LOADER
# =============================================================================
class SequentialBatchSource:
    """
    Flat-frame DALI source.
    group_ids removed because they were unused.
    labels are still repeated per frame so prepare_sequence_batch() keeps working.
    """

    def __init__(
        self,
        sequence_frame_paths,
        sequence_labels,
        batch_size,
        frames_per_window,
        drop_last=True,
        shard_id=0,
        num_shards=1,
    ):
        self.sequence_frame_paths = sequence_frame_paths
        self.sequence_labels = sequence_labels
        self.batch_size = batch_size
        self.frames_per_window = frames_per_window
        self.drop_last = drop_last
        self.shard_id = shard_id
        self.num_shards = num_shards

        dataset_window_indices = list(range(len(self.sequence_labels)))
        self.shard_window_indices = dataset_window_indices[self.shard_id::self.num_shards]

        if self.drop_last:
            usable = (len(self.shard_window_indices) // self.batch_size) * self.batch_size
            self.shard_window_indices = self.shard_window_indices[:usable]
            self.num_batches = len(self.shard_window_indices) // self.batch_size
        else:
            self.num_batches = math.ceil(len(self.shard_window_indices) / self.batch_size)

        self.sequence_frame_paths_encoded = [
            [np.frombuffer(path.encode("utf-8"), dtype=np.uint8) for path in seq_paths]
            for seq_paths in self.sequence_frame_paths
        ]

        self.sequence_repeated_labels = [
            [np.array([label], dtype=np.float32) for _ in range(self.frames_per_window)]
            for label in self.sequence_labels
        ]

        self.cursor = 0

    def reset(self):
        self.cursor = 0

    def __len__(self):
        return self.num_batches

    def __call__(self):
        if self.cursor >= len(self.shard_window_indices):
            raise StopIteration

        batch_indices = self.shard_window_indices[self.cursor:self.cursor + self.batch_size]
        self.cursor += self.batch_size

        if len(batch_indices) < self.batch_size and self.drop_last:
            raise StopIteration

        flat_paths = [path for idx in batch_indices for path in self.sequence_frame_paths_encoded[idx]]
        flat_labels = [label for idx in batch_indices for label in self.sequence_repeated_labels[idx]]

        return flat_paths, flat_labels


class DALISequenceLoader:
    def __init__(self, iterator, batch_source, frames_per_window):
        self.iterator = iterator
        self.batch_source = batch_source
        self.frames_per_window = frames_per_window

    def __iter__(self):
        return iter(self.iterator)

    def __len__(self):
        return len(self.batch_source)

    def reset(self):
        self.batch_source.reset()
        self.iterator.reset()


@pipeline_def
def val_sequence_pipeline(batch_source, image_height, image_width, frames_per_window):
    filepaths, labels = fn.external_source(
        source=batch_source,
        num_outputs=2,
        batch=True,
        parallel=False
    )

    encoded = fn.io.file.read(filepaths)

    images = fn.experimental.decoders.image(
        encoded,
        device="mixed",
        output_type=types.GRAY
    )

    images = fn.resize(images, resize_x=image_width, resize_y=image_height)

    images = fn.crop_mirror_normalize(
        images,
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[127.5],
        std=[127.5],
    )

    labels = fn.cast(labels, dtype=types.FLOAT)
    return images, labels


# =============================================================================
# MODELS
# =============================================================================
class EnhancementBlock(nn.Module):
    def __init__(self, channels=1, hidden_channels=8):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=4, num_channels=hidden_channels),
            nn.GELU(),
            nn.Conv2d(hidden_channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        enhanced = self.block(x)
        return x + 0.1 * enhanced


class CNNEncoder(nn.Module):
    def __init__(self, encoder_backbone_name, encoder_output_dim, encoder_dropout):
        super().__init__()

        self.enhancer = EnhancementBlock(channels=1, hidden_channels=8)

        self.backbone = timm.create_model(
            encoder_backbone_name,
            pretrained=True,
            in_chans=1,
            num_classes=0,
            global_pool=""
        )

        for param in self.backbone.parameters():
            param.requires_grad = True

        if hasattr(self.backbone, "conv_head"):
            for param in self.backbone.conv_head.parameters():
                param.requires_grad = False

        backbone_output_dim = self.backbone.num_features

        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(backbone_output_dim, encoder_output_dim),
            nn.LayerNorm(encoder_output_dim),
            nn.GELU(),
            nn.Dropout(encoder_dropout),
        )

    def forward(self, x):
        x = self.enhancer(x)
        x = self.backbone.forward_features(x)

        if x.ndim == 4:
            x = self.pool(x)
        elif x.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature shape: {x.shape}")

        x = self.fc(x)
        return x


class LSTM(nn.Module):
    def __init__(
        self,
        encoder_backbone_name,
        encoder_output_dim,
        encoder_dropout,
        lstm_hidden_dim,
        num_classes,
        lstm_num_layers,
        temporal_dropout,
        proj_size=0
    ):
        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        self.lstm = nn.LSTM(
            input_size=encoder_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=temporal_dropout if lstm_num_layers > 1 else 0.0,
            proj_size=proj_size,
        )

        final_dim = proj_size if proj_size > 0 else lstm_hidden_dim
        self.classifier = nn.Linear(final_dim * 2, num_classes)

    def forward_features(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)

        lstm_out, _ = self.lstm(features)
        last_timestep = lstm_out[:, -2:, :]
        last_timestep = last_timestep.reshape(B, -1)
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits


class CausalTemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, temporal_dropout, num_groups=8, kernel_size=3):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)

        self.norm1 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm2 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)
        self.norm3 = nn.GroupNorm(num_groups=num_groups, num_channels=out_channels)

        self.activation = nn.GELU()
        self.dropout = nn.Dropout(temporal_dropout)

        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.residual(x)

        out = self.conv1(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm1(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv2(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm2(out)
        out = self.activation(out)
        out = self.dropout(out)

        out = self.conv3(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.norm3(out)
        out = self.activation(out)
        out = self.dropout(out)

        return self.activation(out + residual)


class TemporalConv(nn.Module):
    def __init__(self, encoder_backbone_name, encoder_output_dim, encoder_dropout, temporal_dropout, tcn_hidden_dim, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        self.tcn = nn.Sequential(
            *[
                CausalTemporalConvBlock(
                    in_channels=encoder_output_dim if i == 0 else tcn_hidden_dim[i - 1],
                    out_channels=tcn_hidden_dim[i],
                    dilation=2 ** i,
                    temporal_dropout=temporal_dropout,
                )
                for i in range(len(tcn_hidden_dim))
            ]
        )

        self.classifier = nn.Linear(tcn_hidden_dim[-1], num_classes)

    def forward_features(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features.transpose(1, 2)
        tcn_out = self.tcn(features)
        last_timestep = tcn_out[:, :, -1]
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits


class TransformerEncoder(nn.Module):
    def __init__(
        self,
        encoder_backbone_name,
        encoder_output_dim,
        encoder_dropout,
        temporal_dropout,
        num_heads,
        num_layers,
        dim_feedforward,
        max_len,
        num_classes
    ):
        super().__init__()
        self.num_classes = num_classes
        self.is_binary = num_classes == 1

        self.encoder = CNNEncoder(
            encoder_backbone_name=encoder_backbone_name,
            encoder_output_dim=encoder_output_dim,
            encoder_dropout=encoder_dropout,
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, encoder_output_dim))

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=encoder_output_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=temporal_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)
        self.classifier = nn.Linear(encoder_output_dim, num_classes)

    def _generate_causal_mask(self, seq_len, device, dtype):
        return torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=device, dtype=dtype),
            diagonal=1
        )

    def forward_features(self, x):
        B, T, C, H, W = x.shape

        if T > self.pos_embedding.size(1):
            raise ValueError(f"Sequence length T={T} exceeds max_len={self.pos_embedding.size(1)}")

        x = x.view(B * T, C, H, W)
        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features + self.pos_embedding[:, :T, :]

        causal_mask = self._generate_causal_mask(
            seq_len=T,
            device=features.device,
            dtype=features.dtype,
        )

        transformed = self.transformer(features, mask=causal_mask)
        last_timestep = transformed[:, -1, :]
        return last_timestep

    def forward(self, x):
        last_timestep = self.forward_features(x)
        logits = self.classifier(last_timestep)

        if self.is_binary:
            return logits.squeeze(1)
        return logits


def build_model(config):
    if config["model_name"] == "lstm":
        return LSTM(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            lstm_hidden_dim=config["lstm_hidden_dim"],
            num_classes=config["num_classes"],
            lstm_num_layers=config["lstm_num_layers"],
            temporal_dropout=config["temporal_dropout"],
            proj_size=config.get("proj_size", 0),
        )

    if config["model_name"] == "temporal_conv":
        return TemporalConv(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            tcn_hidden_dim=config["tcn_hidden_dim"],
            num_classes=config["num_classes"],
        )

    if config["model_name"] == "transformer_encoder":
        return TransformerEncoder(
            encoder_backbone_name=config["encoder_backbone_name"],
            encoder_output_dim=config["encoder_output_dim"],
            encoder_dropout=config["encoder_dropout"],
            temporal_dropout=config["temporal_dropout"],
            num_heads=config["transformer_num_heads"],
            num_layers=config["transformer_num_layers"],
            dim_feedforward=config["transformer_hidden_dim"],
            max_len=config["frames_per_window"],
            num_classes=config["num_classes"],
        )

    raise ValueError(f"Unknown model_name: {config['model_name']}")


def get_branch_config(task_name: str) -> dict:
    if task_name == "eye":
        return {
            "model_name": "lstm",
            "encoder_backbone_name": "mobilenetv3_small_100",
            "encoder_output_dim": 288,
            "encoder_dropout": 0.2,
            "temporal_dropout": 0.2,
            "frames_per_window": CONFIG["frames_per_window"],
            "classification_type": "binary",
            "num_classes": 1,
            "lstm_hidden_dim": 288,
            "lstm_num_layers": 2,
            "proj_size": 0,
        }

    if task_name == "mouth":
        return {
            "model_name": "lstm",
            "encoder_backbone_name": "mobilenetv3_small_100",
            "encoder_output_dim": 288,
            "encoder_dropout": 0.2,
            "temporal_dropout": 0.2,
            "frames_per_window": CONFIG["frames_per_window"],
            "classification_type": "multiclass",
            "num_classes": 3,
            "lstm_hidden_dim": 288,
            "lstm_num_layers": 2,
            "proj_size": 0,
        }

    if task_name == "head":
        return {
            "model_name": "lstm",
            "encoder_backbone_name": "mobilenetv3_small_100",
            "encoder_output_dim": 288,
            "encoder_dropout": 0.2,
            "temporal_dropout": 0.2,
            "frames_per_window": CONFIG["frames_per_window"],
            "classification_type": "multiclass",
            "num_classes": 3,
            "lstm_hidden_dim": 288,
            "lstm_num_layers": 2,
            "proj_size": 0,
        }

    raise ValueError(f"Unknown task_name: {task_name}")


# =============================================================================
# FUSION DATASET / METRICS / FUSION MODEL
# =============================================================================
class DrowsinessFusionDataset(Dataset):
    def __init__(self, head_npz_path, eye_npz_path, mouth_npz_path):
        head_data = np.load(head_npz_path)
        eye_data = np.load(eye_npz_path)
        mouth_data = np.load(mouth_npz_path)

        assert np.array_equal(head_data["ids"], eye_data["ids"])
        assert np.array_equal(head_data["ids"], mouth_data["ids"])
        assert np.array_equal(head_data["labels"], eye_data["labels"])
        assert np.array_equal(head_data["labels"], mouth_data["labels"])

        self.ids = head_data["ids"]
        self.labels = head_data["labels"].astype(np.float32)

        head_x = head_data["features"]
        eye_x = eye_data["features"]
        mouth_x = mouth_data["features"]

        self.features = np.concatenate([head_x, eye_x, mouth_x], axis=1).astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        sample_id = torch.tensor(self.ids[idx], dtype=torch.long)
        return x, y, sample_id












class FusionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)















def compute_binary_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def evaluate_fusion_model(model, loader, criterion, threshold):
    model.eval()

    total_loss = 0.0
    total_count = 0

    all_ids = []
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y, sample_ids in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, y)

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds = (probs >= threshold).astype(np.int64)
            labels = y.detach().cpu().numpy().astype(np.int64)

            bs = y.size(0)
            total_loss += loss.item() * bs
            total_count += bs

            all_ids.append(sample_ids.numpy())
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels)

    all_ids = np.concatenate(all_ids, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    print("val positive preds:", int(all_preds.sum()))
    print("val positive labels:", int(all_labels.sum()))

    tp = int(((all_preds == 1) & (all_labels == 1)).sum())
    tn = int(((all_preds == 0) & (all_labels == 0)).sum())
    fp = int(((all_preds == 1) & (all_labels == 0)).sum())
    fn = int(((all_preds == 0) & (all_labels == 1)).sum())
    print(f"TP={tp} TN={tn} FP={fp} FN={fn}")

    avg_loss = total_loss / max(total_count, 1)
    metrics = compute_binary_metrics(all_labels, all_preds)
    return avg_loss, metrics, all_ids, all_probs, all_preds, all_labels


def search_best_threshold(model, loader, threshold_candidates=None):
    if threshold_candidates is None:
        threshold_candidates = [0.40, 0.45, 0.47, 0.50, 0.52, 0.55, 0.60]

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            labels = y.numpy().astype(np.int64)

            all_probs.append(probs)
            all_labels.append(labels)

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    best_threshold = 0.5
    best_acc = -1.0
    best_metrics = None

    for th in threshold_candidates:
        preds = (all_probs >= th).astype(np.int64)
        metrics = compute_binary_metrics(all_labels, preds)

        if metrics["accuracy"] > best_acc:
            best_acc = metrics["accuracy"]
            best_threshold = th
            best_metrics = metrics

    return best_threshold, best_metrics


def make_fusion_loader(split_npzs, shuffle):
    ds = DrowsinessFusionDataset(
        head_npz_path=split_npzs["head"],
        eye_npz_path=split_npzs["eye"],
        mouth_npz_path=split_npzs["mouth"],
    )
    loader = DataLoader(
        ds,
        batch_size=CONFIG["fusion_batch_size"],
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
    )
    return ds, loader


def train_fusion_model(exported_npzs):
    train_ds, train_loader = make_fusion_loader(exported_npzs["train"], shuffle=True)
    val_ds, val_loader = make_fusion_loader(exported_npzs["val"], shuffle=False)

    input_dim = train_ds.features.shape[1]
    model = FusionMLP(input_dim=input_dim).to(device)

    train_labels = train_ds.labels
    pos_count = float((train_labels == 1).sum())
    neg_count = float((train_labels == 0).sum())
    pos_weight_value = neg_count / max(pos_count, 1.0)

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=CONFIG["fusion_lr"],
        weight_decay=CONFIG["fusion_weight_decay"],
    )

    best_val_acc = -1.0
    best_threshold = 0.5
    best_epoch = -1
    patience_counter = 0

    best_model_path = os.path.join(PATHS["output_dir"], "best_drowsiness_fusion_model.pth")
    log_csv_path = os.path.join(PATHS["output_dir"], "fusion_train_log.csv")

    with open(log_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "val_loss", "best_threshold",
            "val_accuracy", "val_precision", "val_recall", "val_f1",
        ])

        for epoch in range(CONFIG["fusion_epochs"]):
            model.train()
            total_train_loss = 0.0
            total_train_count = 0

            for x, y, _ in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad()
                logits = model(x)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

                bs = y.size(0)
                total_train_loss += loss.item() * bs
                total_train_count += bs

            avg_train_loss = total_train_loss / max(total_train_count, 1)
            epoch_best_threshold, _ = search_best_threshold(model, val_loader)

            val_loss, val_metrics, _, _, _, _ = evaluate_fusion_model(
                model=model,
                loader=val_loader,
                criterion=criterion,
                threshold=epoch_best_threshold,
            )

            if accelerator.is_main_process:
                print(
                    f"Epoch {epoch + 1}/{CONFIG['fusion_epochs']} | "
                    f"train_loss={avg_train_loss:.4f} | "
                    f"val_loss={val_loss:.4f} | "
                    f"th={epoch_best_threshold:.2f} | "
                    f"val_acc={val_metrics['accuracy']:.4f} | "
                    f"val_f1={val_metrics['f1']:.4f}"
                )

            writer.writerow([
                epoch + 1,
                f"{avg_train_loss:.6f}",
                f"{val_loss:.6f}",
                f"{epoch_best_threshold:.4f}",
                f"{val_metrics['accuracy']:.6f}",
                f"{val_metrics['precision']:.6f}",
                f"{val_metrics['recall']:.6f}",
                f"{val_metrics['f1']:.6f}",
            ])
            f.flush()

            if val_metrics["accuracy"] > best_val_acc:
                best_val_acc = val_metrics["accuracy"]
                best_threshold = epoch_best_threshold
                best_epoch = epoch + 1
                patience_counter = 0

                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "input_dim": input_dim,
                        "best_threshold": best_threshold,
                        "best_val_accuracy": best_val_acc,
                        "epoch": best_epoch,
                        "use_features": CONFIG["use_features"],
                    },
                    best_model_path,
                )

                if accelerator.is_main_process:
                    print(
                        f"Saved best fusion model | epoch={best_epoch} "
                        f"| val_acc={best_val_acc:.4f} | th={best_threshold:.2f}"
                    )
            else:
                patience_counter += 1
                if patience_counter >= CONFIG["fusion_patience"]:
                    if accelerator.is_main_process:
                        print("Early stopping triggered.")
                    break

    if accelerator.is_main_process:
        print("\nFusion training finished.")
        print(f"Best epoch     : {best_epoch}")
        print(f"Best val acc   : {best_val_acc:.4f}")
        print(f"Best threshold : {best_threshold:.4f}")

    return best_model_path


def run_fusion_test(exported_npzs, fusion_checkpoint_path):
    test_ds, test_loader = make_fusion_loader(exported_npzs["test"], shuffle=False)

    checkpoint = torch.load(fusion_checkpoint_path, map_location=device)

    model = FusionMLP(input_dim=checkpoint["input_dim"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    threshold = checkpoint.get("best_threshold", 0.5)
    criterion = nn.BCEWithLogitsLoss()

    test_loss, test_metrics, all_ids, all_probs, all_preds, all_labels = evaluate_fusion_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        threshold=threshold,
    )

    out_npz = os.path.join(PATHS["output_dir"], "test_drowsiness_predictions.npz")
    out_csv = os.path.join(PATHS["output_dir"], "test_drowsiness_predictions.csv")

    np.savez(
        out_npz,
        ids=all_ids,
        probs=all_probs,
        preds=all_preds,
        labels=all_labels,
    )

    with open(out_csv, "w", encoding="utf-8") as f:
        f.write("id,prob,pred,label\n")
        for sid, prob, pred, label in zip(all_ids, all_probs, all_preds, all_labels):
            f.write(f"{sid},{float(prob):.8f},{int(pred)},{int(label)}\n")

    if accelerator.is_main_process:
        print("\nFusion test results")
        print(f"threshold : {threshold:.4f}")
        print(f"loss      : {test_loss:.4f}")
        print(f"accuracy  : {test_metrics['accuracy']:.4f}")
        print(f"precision : {test_metrics['precision']:.4f}")
        print(f"recall    : {test_metrics['recall']:.4f}")
        print(f"f1        : {test_metrics['f1']:.4f}")
        print(f"Saved npz : {out_npz}")
        print(f"Saved csv : {out_csv}")


# =============================================================================
# FAST CACHED INDEX (LEAN VERSION, CHECKS REMOVED)
# =============================================================================
class DDDSequenceIndex:
    """
    Lean cached sequence index.

    Removed from traversal path:
    - is_dir() checks
    - missing-root skipping
    - sorted(...) traversal
    - zero-length / short-video protective checks in directory walk stage

    Kept:
    - drowsiness txt resolution logic
    - frame bound checks where required for correctness
    - cache save/load
    - clone_for_new_roots for reuse across eye/mouth/head
    """

    def __init__(
        self,
        image_root_dirs,
        time_window_in_seconds,
        frames_per_window,
        window_stride_by_seconds,
        label_source="filename",
        drowsiness_label_root_dirs=None,
        cache_path=None,
        force_rebuild=False,
    ):
        if isinstance(image_root_dirs, str):
            image_root_dirs = [image_root_dirs]
        self.image_root_dirs = [p for p in image_root_dirs if p]

        if drowsiness_label_root_dirs is None:
            drowsiness_label_root_dirs = []
        elif isinstance(drowsiness_label_root_dirs, str):
            drowsiness_label_root_dirs = [drowsiness_label_root_dirs]
        self.drowsiness_label_root_dirs = [p for p in drowsiness_label_root_dirs if p]

        self.time_window_in_seconds = time_window_in_seconds
        self.frames_per_window = frames_per_window
        self.window_stride_by_seconds = window_stride_by_seconds
        self.label_source = label_source

        self.class_counts = {}
        self.sequence_labels = []
        self.sequence_group_keys = []
        self.group_to_id = {}
        self.id_to_group = {}

        self.sequence_root_indices = []
        self.sequence_relative_frame_paths = []
        self.sequence_frame_paths = []

        self.cache_path = cache_path

        if cache_path and os.path.exists(cache_path) and not force_rebuild:
            self._load_cache(cache_path)
            self._materialize_absolute_paths(self.image_root_dirs)
            print(f"[INDEX CACHE] loaded: {cache_path}")
        else:
            self._build_sequences()
            self._materialize_absolute_paths(self.image_root_dirs)
            if cache_path:
                os.makedirs(os.path.dirname(cache_path), exist_ok=True)
                self._save_cache(cache_path)
                print(f"[INDEX CACHE] saved: {cache_path}")

    def _save_cache(self, cache_path):
        payload = {
            "time_window_in_seconds": self.time_window_in_seconds,
            "frames_per_window": self.frames_per_window,
            "window_stride_by_seconds": self.window_stride_by_seconds,
            "label_source": self.label_source,
            "class_counts": self.class_counts,
            "sequence_labels": self.sequence_labels,
            "sequence_group_keys": self.sequence_group_keys,
            "group_to_id": self.group_to_id,
            "id_to_group": self.id_to_group,
            "sequence_root_indices": self.sequence_root_indices,
            "sequence_relative_frame_paths": self.sequence_relative_frame_paths,
        }

        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(payload, f)

    def _load_cache(self, cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        self.class_counts = {int(k): int(v) for k, v in payload["class_counts"].items()}
        self.sequence_labels = [int(x) for x in payload["sequence_labels"]]
        self.sequence_group_keys = list(payload["sequence_group_keys"])
        self.group_to_id = {str(k): int(v) for k, v in payload["group_to_id"].items()}
        self.id_to_group = {int(k): str(v) for k, v in payload["id_to_group"].items()}
        self.sequence_root_indices = [int(x) for x in payload["sequence_root_indices"]]
        self.sequence_relative_frame_paths = [list(x) for x in payload["sequence_relative_frame_paths"]]

    def _materialize_absolute_paths(self, image_root_dirs):
        self.sequence_frame_paths = []
        for root_idx, rel_seq_paths in zip(self.sequence_root_indices, self.sequence_relative_frame_paths):
            root_dir = image_root_dirs[root_idx]
            self.sequence_frame_paths.append([os.path.join(root_dir, rel_path) for rel_path in rel_seq_paths])

    def clone_for_new_roots(self, new_image_root_dirs):
        cloned = DDDSequenceIndex.__new__(DDDSequenceIndex)

        if isinstance(new_image_root_dirs, str):
            new_image_root_dirs = [new_image_root_dirs]
        cloned.image_root_dirs = [p for p in new_image_root_dirs if p]

        cloned.drowsiness_label_root_dirs = list(self.drowsiness_label_root_dirs)
        cloned.time_window_in_seconds = self.time_window_in_seconds
        cloned.frames_per_window = self.frames_per_window
        cloned.window_stride_by_seconds = self.window_stride_by_seconds
        cloned.label_source = self.label_source

        cloned.class_counts = dict(self.class_counts)
        cloned.sequence_labels = list(self.sequence_labels)
        cloned.sequence_group_keys = list(self.sequence_group_keys)
        cloned.group_to_id = dict(self.group_to_id)
        cloned.id_to_group = dict(self.id_to_group)

        cloned.sequence_root_indices = list(self.sequence_root_indices)
        cloned.sequence_relative_frame_paths = [list(x) for x in self.sequence_relative_frame_paths]
        cloned.cache_path = self.cache_path

        cloned._materialize_absolute_paths(cloned.image_root_dirs)
        return cloned

    def _build_sequences(self):
        for root_idx, image_root_dir in enumerate(self.image_root_dirs):
            print(f"[INDEX] scanning root: {image_root_dir}")

            subject_count = 0
            for subject_entry in os.scandir(image_root_dir):
                subject_count += 1

                if subject_count % 5 == 0:
                    print(
                        f"[INDEX] root={image_root_dir} | subjects_done={subject_count} | "
                        f"sequences_so_far={len(self.sequence_relative_frame_paths)}"
                    )

                subject_name = subject_entry.name

                for scenario_entry in os.scandir(subject_entry.path):
                    scenario_name = scenario_entry.name
                    video_fps = 15 if "night" in scenario_name.lower() else 30

                    for video_type_entry in os.scandir(scenario_entry.path):
                        self._process_windows_in_video(
                            root_idx=root_idx,
                            image_root_dir=image_root_dir,
                            video_path=video_type_entry.path,
                            video_fps=video_fps,
                            subject_name=subject_name,
                            scenario_name=scenario_name,
                            video_type_name=video_type_entry.name,
                        )

            print(f"[INDEX DONE] root={image_root_dir} | total_sequences={len(self.sequence_relative_frame_paths)}")

    def _process_windows_in_video(
        self,
        root_idx,
        image_root_dir,
        video_path,
        video_fps,
        subject_name,
        scenario_name,
        video_type_name,
    ):
        video_frame_list = sorted(entry.name for entry in os.scandir(video_path))
        total_frames_in_video = len(video_frame_list)

        available_frames_per_window = round(self.time_window_in_seconds * video_fps)
        stride_between_windows_in_frames = max(1, round(self.window_stride_by_seconds * video_fps))
        total_windows_in_video = (total_frames_in_video - available_frames_per_window) // stride_between_windows_in_frames + 1

        if self.frames_per_window == 1:
            selected_frames_indices_in_window = [0]
        else:
            frames_space_in_window = (available_frames_per_window - 1) / (self.frames_per_window - 1)
            selected_frames_indices_in_window = [
                round(frame_position * frames_space_in_window)
                for frame_position in range(self.frames_per_window)
            ]

        last_frame_offset = selected_frames_indices_in_window[-1]

        relative_video_dir = os.path.relpath(video_path, image_root_dir)
        relative_frame_paths = [os.path.join(relative_video_dir, name) for name in video_frame_list]

        if self.label_source == "filename":
            frame_labels = [self._label_from_filename(name) for name in video_frame_list]
        elif self.label_source == "drowsiness_txt":
            frame_labels = self._load_drowsiness_labels(
                subject_name=subject_name,
                scenario_name=scenario_name,
                video_type_name=video_type_name,
                frame_names=video_frame_list,
            )
        else:
            raise ValueError(f"Unsupported label_source: {self.label_source}")

        append_rel_paths = self.sequence_relative_frame_paths.append
        append_root_idx = self.sequence_root_indices.append
        append_label = self.sequence_labels.append
        append_group = self.sequence_group_keys.append
        class_counts = self.class_counts

        group_key = f"{subject_name} | {scenario_name}"

        if group_key not in self.group_to_id:
            group_id = len(self.group_to_id)
            self.group_to_id[group_key] = group_id
            self.id_to_group[group_id] = group_key

        for window_index in range(total_windows_in_video):
            window_start_frame = window_index * stride_between_windows_in_frames

            last_frame_index = window_start_frame + last_frame_offset
            if last_frame_index >= total_frames_in_video:
                break

            frame_indices = [window_start_frame + offset for offset in selected_frames_indices_in_window]

            for i in range(1, len(frame_indices)):
                if frame_indices[i] <= frame_indices[i - 1]:
                    frame_indices[i] = frame_indices[i - 1] + 1

            if frame_indices[-1] >= total_frames_in_video:
                break

            rel_paths = [relative_frame_paths[idx] for idx in frame_indices]
            label = frame_labels[frame_indices[-1]]

            class_counts[label] = class_counts.get(label, 0) + 1
            append_rel_paths(rel_paths)
            append_root_idx(root_idx)
            append_label(label)
            append_group(group_key)

    def _label_from_filename(self, filename):
        return int(filename.rsplit("_", 1)[-1].split(".")[0])

    def _extract_frame_number_from_filename(self, filename):
        stem = os.path.splitext(os.path.basename(filename))[0]
        parts = stem.split("_")
        numeric_parts = [part for part in parts if part.isdigit()]

        if len(numeric_parts) >= 4:
            return int(numeric_parts[1])

        raise ValueError(f"Unexpected filename format: {filename}")

    def _find_drowsiness_label_file(self, subject_name, scenario_name, video_type_name):
        if not self.drowsiness_label_root_dirs:
            raise ValueError("drowsiness_label_root_dirs must be provided when label_source='drowsiness_txt'")

        scenario_aliases = [scenario_name]
        normalized_scenario = scenario_name.lower().replace("_", "").replace("-", "")

        if normalized_scenario == "nightnoglasses":
            scenario_aliases.extend(["nightnoglasses", "night_noglasses", "night-no-glasses"])
        elif normalized_scenario == "nightglasses":
            scenario_aliases.extend(["nightglasses", "night_glasses", "night-glasses"])
        elif normalized_scenario == "noglasses":
            scenario_aliases.extend(["noglasses", "no_glasses", "no-glasses"])

        scenario_aliases = list(dict.fromkeys(scenario_aliases))

        video_type_aliases = [video_type_name]
        normalized_video_type = video_type_name.lower().replace("_", "").replace("-", "")

        if normalized_video_type in ("mix", "mixing"):
            video_type_aliases.extend(["mix", "mixing"])
        elif normalized_video_type in ("nonsleepycombination",):
            video_type_aliases.extend(["nonsleepyCombination"])
        elif normalized_video_type in ("sleepycombination",):
            video_type_aliases.extend(["sleepyCombination"])

        video_type_aliases = list(dict.fromkeys(video_type_aliases))

        all_candidates = []

        for label_root_dir in self.drowsiness_label_root_dirs:
            search_dirs = []

            for scenario_alias in scenario_aliases:
                search_dirs.append(os.path.join(label_root_dir, subject_name, scenario_alias))

            search_dirs.append(os.path.join(label_root_dir, subject_name, "labels"))
            search_dirs.append(os.path.join(label_root_dir, subject_name))

            seen_dirs = set()
            deduped_search_dirs = []
            for d in search_dirs:
                if d not in seen_dirs:
                    deduped_search_dirs.append(d)
                    seen_dirs.add(d)

            for search_dir in deduped_search_dirs:
                if not os.path.isdir(search_dir):
                    continue

                for name in os.listdir(search_dir):
                    lower_name = name.lower()
                    expected_names = set()

                    for scenario_alias in scenario_aliases:
                        for vt_alias in video_type_aliases:
                            expected_names.add(f"{subject_name}_{scenario_alias}_{vt_alias}_drowsiness.txt".lower())
                            expected_names.add(f"{subject_name}_{vt_alias}_drowsiness.txt".lower())

                    if lower_name in expected_names:
                        all_candidates.append(os.path.join(search_dir, name))

        all_candidates = sorted(set(all_candidates))

        if len(all_candidates) == 1:
            return all_candidates[0]

        if len(all_candidates) == 0:
            raise FileNotFoundError(
                f"No drowsiness txt found for subject={subject_name}, "
                f"scenario={scenario_name}, video_type={video_type_name}"
            )

        raise RuntimeError(
            f"Multiple matching drowsiness txt files found for subject={subject_name}, "
            f"scenario={scenario_name}, video_type={video_type_name}:\n" + "\n".join(all_candidates)
        )

    def _read_label_file(self, label_file_path):
        with open(label_file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        content = content.replace(" ", "").replace("\n", "").replace("\r", "").replace("\t", "")
        labels = [int(ch) for ch in content]
        return labels

    def _load_drowsiness_labels(self, subject_name, scenario_name, video_type_name, frame_names):
        label_file_path = self._find_drowsiness_label_file(
            subject_name=subject_name,
            scenario_name=scenario_name,
            video_type_name=video_type_name,
        )

        txt_labels = self._read_label_file(label_file_path)
        frame_labels = []

        for frame_name in frame_names:
            frame_number = self._extract_frame_number_from_filename(frame_name)

            if frame_number < 0 or frame_number >= len(txt_labels):
                raise IndexError(
                    f"Frame number {frame_number} out of range for label file {label_file_path} "
                    f"(number of labels = {len(txt_labels)})"
                )

            frame_labels.append(txt_labels[frame_number])

        return frame_labels




def build_index_cache_path(split_name, task_name, input_dirs, label_root_dirs):
    cache_dir = os.path.join(PATHS["output_dir"], "sequence_index_cache")
    os.makedirs(cache_dir, exist_ok=True)

    payload = {
        "split_name": split_name,
        "task_name": task_name,
        "input_dirs": list(input_dirs),
        "label_root_dirs": list(label_root_dirs),
        "time_window_in_seconds": CONFIG["time_window_in_seconds"],
        "frames_per_window": CONFIG["frames_per_window"],
        "window_stride_by_seconds": CONFIG["window_stride_by_seconds"],
        "label_source": "drowsiness_txt",
    }

    payload_str = json.dumps(payload, sort_keys=True)
    payload_hash = hashlib.md5(payload_str.encode("utf-8")).hexdigest()
    return os.path.join(cache_dir, f"{split_name}_{task_name}_{payload_hash}.json")


# =============================================================================
# EXPORT
# =============================================================================
def create_export_dali_loader(
    accelerator,
    export_index,
    batch_size,
    prefetch_queue_depth,
    image_height,
    image_width,
):
    device_id = accelerator.local_process_index
    shard_id = accelerator.local_process_index
    num_shards = accelerator.num_processes

    print("[LOADER] sequence index ready")

    export_source = SequentialBatchSource(
        sequence_frame_paths=export_index.sequence_frame_paths,
        sequence_labels=export_index.sequence_labels,
        batch_size=batch_size,
        frames_per_window=CONFIG["frames_per_window"],
        drop_last=False,
        shard_id=shard_id,
        num_shards=num_shards,
    )

    print("[LOADER] batch source built")

    export_pipe = val_sequence_pipeline(
        batch_source=export_source,
        image_height=image_height,
        image_width=image_width,
        frames_per_window=CONFIG["frames_per_window"],
        batch_size=batch_size * CONFIG["frames_per_window"],
        num_threads=4,
        device_id=device_id,
        seed=seed + shard_id,
        prefetch_queue_depth=prefetch_queue_depth,
    )

    print("[LOADER] building DALI pipeline")
    export_pipe.build()
    print("[LOADER] DALI pipeline built")

    export_iter = DALIGenericIterator(
        [export_pipe],
        output_map=["frames", "labels"],
        size=-1,
        auto_reset=False,
        prepare_first_batch=True,
    )
    print("[LOADER] DALI iterator ready")

    export_loader = DALISequenceLoader(export_iter, export_source, CONFIG["frames_per_window"])
    return export_loader


def export_branch_features(
    model,
    loader,
    accelerator,
    frames_per_window,
    classification_type,
    output_path,
):
    model.eval()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    tmp_feature_path = output_path.replace(".npz", "_features_tmp.npy")
    tmp_label_path = output_path.replace(".npz", "_labels_tmp.npy")
    tmp_id_path = output_path.replace(".npz", "_ids_tmp.npy")

    total_sequences = len(loader.batch_source.shard_window_indices)

    feature_mm = None
    label_mm = None
    id_mm = None
    write_offset = 0

    with torch.inference_mode():
        for batch_idx, data in enumerate(loader):
            batch = data[0]
            frames = batch["frames"]
            labels = batch["labels"]

            frames, labels = prepare_sequence_batch(
                frames=frames,
                labels=labels,
                frames_per_window=frames_per_window,
                classification_type=classification_type,
            )

            labels = labels.view(-1).long()
            features = model.forward_features(frames)

            features_np = features.detach().cpu().to(torch.float16).numpy()
            labels_np = labels.detach().cpu().numpy().astype(np.int64)

            current_bs = labels_np.shape[0]

            if feature_mm is None:
                feature_dim = features_np.shape[1]

                feature_mm = open_memmap(
                    tmp_feature_path,
                    mode="w+",
                    dtype=np.float16,
                    shape=(total_sequences, feature_dim),
                )
                label_mm = open_memmap(
                    tmp_label_path,
                    mode="w+",
                    dtype=np.int64,
                    shape=(total_sequences,),
                )
                id_mm = open_memmap(
                    tmp_id_path,
                    mode="w+",
                    dtype=np.int64,
                    shape=(total_sequences,),
                )

            feature_mm[write_offset:write_offset + current_bs] = features_np
            label_mm[write_offset:write_offset + current_bs] = labels_np
            id_mm[write_offset:write_offset + current_bs] = np.arange(
                write_offset,
                write_offset + current_bs,
                dtype=np.int64,
            )

            write_offset += current_bs

            if (batch_idx + 1) % 20 == 0:
                print(f"[EXPORT] batch={batch_idx + 1} | written={write_offset}/{total_sequences}")

    if accelerator.is_main_process:
        all_features = np.asarray(feature_mm[:write_offset])
        all_labels = np.asarray(label_mm[:write_offset])
        all_ids = np.asarray(id_mm[:write_offset])

        np.savez(
            output_path,
            ids=all_ids,
            features=all_features,
            labels=all_labels,
        )

        unique_vals, counts = np.unique(all_labels, return_counts=True)
        print("FINAL LABEL COUNTS:", dict(zip(unique_vals.tolist(), counts.tolist())))
        print("TOTAL POSITIVES:", int((all_labels == 1).sum()))
        print("TOTAL SAMPLES:", len(all_labels))
        print(f"✅ Saved branch embeddings → {output_path}")

    for tmp_path in [tmp_feature_path, tmp_label_path, tmp_id_path]:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def should_skip_existing_npz(output_npz_path):
    return os.path.exists(output_npz_path) and os.path.getsize(output_npz_path) > 0




def export_one_branch(task_name, split_name, input_dirs, checkpoint_path, output_npz_path, drowsiness_label_root_dirs):
    print("\n" + "=" * 90)
    print(f"EXPORT {task_name.upper()} | split={split_name}")
    print(f"input_dirs  : {input_dirs}")
    print(f"checkpoint  : {checkpoint_path}")
    print(f"output_npz  : {output_npz_path}")
    print("=" * 90)

    if should_skip_existing_npz(output_npz_path):
        print(f"⏭️ Skipping existing NPZ: {output_npz_path}")
        return output_npz_path

    cache_path = build_index_cache_path(
        split_name=split_name,
        task_name=task_name,
        input_dirs=input_dirs,
        label_root_dirs=drowsiness_label_root_dirs,
    )

    export_index = DDDSequenceIndex(
        image_root_dirs=input_dirs,
        time_window_in_seconds=CONFIG["time_window_in_seconds"],
        frames_per_window=CONFIG["frames_per_window"],
        window_stride_by_seconds=CONFIG["window_stride_by_seconds"],
        label_source="drowsiness_txt",
        drowsiness_label_root_dirs=drowsiness_label_root_dirs,
        cache_path=cache_path,
        force_rebuild=False,
    )

    branch_config = get_branch_config(task_name)
    model = build_model(branch_config)

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    if "best_threshold" in checkpoint:
        print(f"Loaded checkpoint. Stored best threshold = {checkpoint['best_threshold']:.4f}")
    else:
        print("Loaded checkpoint.")

    model = accelerator.prepare(model)

    loader = create_export_dali_loader(
        accelerator=accelerator,
        export_index=export_index,
        batch_size=CONFIG["batch_size"],
        prefetch_queue_depth=CONFIG["prefetch_queue_depth"],
        image_height=CONFIG["image_height"],
        image_width=CONFIG["image_width"],
    )

    export_branch_features(
        model=model,
        loader=loader,
        accelerator=accelerator,
        frames_per_window=CONFIG["frames_per_window"],
        classification_type="binary",
        output_path=output_npz_path,
    )

    return output_npz_path







def export_all_branch_features(include_test=False):
    os.makedirs(PATHS["output_dir"], exist_ok=True)

    exported = {
        "train": {},
        "val": {},
    }

    if include_test:
        exported["test"] = {}

    branch_info = {
        "eye": {
            "ckpt": PATHS["eye_ckpt"],
            "train_dirs": PATHS["eye_train_dirs"],
            "val_dirs": PATHS["eye_val_dirs"],
            "test_dirs": PATHS["eye_test_dirs"],
        },
        "mouth": {
            "ckpt": PATHS["mouth_ckpt"],
            "train_dirs": PATHS["mouth_train_dirs"],
            "val_dirs": PATHS["mouth_val_dirs"],
            "test_dirs": PATHS["mouth_test_dirs"],
        },
        "head": {
            "ckpt": PATHS["head_ckpt"],
            "train_dirs": PATHS["head_train_dirs"],
            "val_dirs": PATHS["head_val_dirs"],
            "test_dirs": PATHS["head_test_dirs"],
        },
    }

    split_names = ["train", "val"]
    if include_test:
        split_names.append("test")

    for split_name in split_names:
        if split_name == "train":
            drowsiness_label_root_dirs = PATHS["drowsiness_train_label_root_dirs"]
        elif split_name == "val":
            drowsiness_label_root_dirs = PATHS["drowsiness_val_label_root_dirs"]
        elif split_name == "test":
            drowsiness_label_root_dirs = PATHS["drowsiness_test_label_root_dirs"]
        else:
            raise ValueError(f"Unknown split_name: {split_name}")

        for task_name, info in branch_info.items():
            input_dirs = info[f"{split_name}_dirs"]
            checkpoint_path = info["ckpt"]
            output_npz_path = os.path.join(PATHS["output_dir"], f"{split_name}_{task_name}.npz")

            export_one_branch(
                task_name=task_name,
                split_name=split_name,
                input_dirs=input_dirs,
                checkpoint_path=checkpoint_path,
                output_npz_path=output_npz_path,
                drowsiness_label_root_dirs=drowsiness_label_root_dirs,
            )

            exported[split_name][task_name] = output_npz_path

    return exported














# =============================================================================
# MAIN
# =============================================================================
def main():
    exported_npzs = export_all_branch_features()
    best_fusion_ckpt = train_fusion_model(exported_npzs)
    # run_fusion_test(exported_npzs, best_fusion_ckpt)


if __name__ == "__main__":
    main()


EXPORT EYE | split=train
input_dirs  : ['/kaggle/input/datasets/manith04/ddd-train1-righteye-v2', '/kaggle/input/datasets/manith04/ddd-train2-righteye-v2']
checkpoint  : /kaggle/input/models/manith04/ddd-manith-bbb/pytorch/default/1/best_val_accuracy_righteye_model.pth
output_npz  : /kaggle/working/abc/fusion_outputs/train_eye.npz
⏭️ Skipping existing NPZ: /kaggle/working/abc/fusion_outputs/train_eye.npz

EXPORT MOUTH | split=train
input_dirs  : ['/kaggle/input/datasets/manith04/ddd-train1-mouth-v2', '/kaggle/input/datasets/manith04/ddd-train2-mouth-v2']
checkpoint  : /kaggle/input/models/manith04/ddd-manith-bbb/pytorch/default/1/best_val_accuracy_mouth_model.pth
output_npz  : /kaggle/working/abc/fusion_outputs/train_mouth.npz
⏭️ Skipping existing NPZ: /kaggle/working/abc/fusion_outputs/train_mouth.npz

EXPORT HEAD | split=train
input_dirs  : ['/kaggle/input/datasets/manith04/ddd-train1-head-v2', '/kaggle/input/datasets/manith04/ddd-train2-head-v2']
checkpoint  : /kaggle/input/models/

In [ ]:
!ls